# Ensemble Learing

| Aspect                        | Simple Voting/Averaging                    | Bagging                                   | Boosting                                               | Stacking                                              |
| ----------------------------- | ------------------------------------------ | ----------------------------------------- | ------------------------------------------------------ | ----------------------------------------------------- |
| **Category**                  | Basic Ensemble                             | Variance Reduction                        | Bias Reduction                                         | Meta-Learning                                         |
| **Training Process**          | Parallel (independent models)              | Parallel (bootstrap samples)              | Sequential (error-focused)                             | Two-stage (base + meta-learner)                       |
| **Base Learners**             | Heterogeneous or homogeneous               | Usually homogeneous (e.g., deep trees)    | Homogeneous weak learners (e.g., shallow trees/stumps) | Heterogeneous (diverse algorithms)                    |
| **Aggregation Method**        | Majority vote (classif.) / Average (regr.) | Majority vote / Average                   | Weighted vote / Sum (weights from performance)         | Learned meta-model (e.g., logistic reg, another tree) |
| **Parallelizable**            | Yes                                        | Yes                                       | No                                                     | Partially (bases yes, meta no)                        |
| **Primary Effect**            | Reduces variance                           | Reduces variance                          | Reduces bias (and variance iteratively)                | Reduces both bias and variance                        |
| **Overfitting Risk**          | Low                                        | Low (especially with many trees)          | High without regularization                            | High (requires careful CV to prevent leakage)         |
| **Noise/Outlier Sensitivity** | Medium                                     | Low                                       | High                                                   | Medium                                                |
| **Best Suited For**           | Quick baseline with diverse models         | High-variance base learners (e.g., trees) | Weak learners to build strong predictor                | Maximizing performance with diverse models            |
| **Computational Cost**        | Low                                        | Medium (sampling + many models)           | High (sequential)                                      | Very High (CV folds + meta)                           |
| **Strengths**                 | Simple, interpretable, fast                | Robust, handles overfitting well          | Very high accuracy, feature importance                 | Often highest accuracy                                |
| **Weaknesses**                | Limited gain if models correlated          | Less effective on biased base learners    | Sensitive to noise, slower training                    | Complex, prone to overfitting, hard to tune           |
| **Use Cases**                 | VotingClassifier in sklearn                | BaggingClassifier, Random Forest          | AdaBoost, Gradient Boosting                            | StackedGeneralization (mlxtend, custom)               |

## Types of Ensemble Learning

### Bagging

```mermaid
flowchart LR
    A(Original Dataset D) e1@--> B1(Bootstrap Sample 1)
    A e2@--> B2(Bootstrap Sample 2)
    A e3@--> B3(Bootstrap Sample 3)
    B1 e4@--> C1(Train Model 1)
    B2 e5@--> C2(Train Model 2)
    B3 e6@--> C3(Train Model 3)
    C1 e7@--> D(Aggregate Predictions<br/>Majority Vote / Average)
    C2 e8@--> D
    C3 e9@--> D
    D e10@--> E(Final Prediction)

    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }

```

### Gradient Boosting Implementations

| Feature                        | GradientBoostingClassifier/Regressor (sklearn) | XGBoost                                     | LightGBM                                   | CatBoost                                        |
| ------------------------------ | ---------------------------------------------- | ------------------------------------------- | ------------------------------------------ | ----------------------------------------------- |
| **Tree Growth Strategy**       | Level-wise (depth-first)                       | Level-wise + approximate histogram          | Leaf-wise (best-first)                     | Symmetric (ordered boosting)                    |
| **Speed**                      | Slow (exact greedy splits)                     | Fast (histogram binning, parallelism)       | Very Fast (GOSS, EFB, histogram)           | Fast (GPU excellent, CPU good)                  |
| **Memory Efficiency**          | High usage                                     | Medium (histogram caching)                  | Low (smaller histograms, gradient binning) | Medium                                          |
| **Scalability**                | Poor (>100k rows slow)                         | Good (out-of-core, distributed)             | Excellent (handles millions easily)        | Very Good                                       |
| **Native Categorical Support** | No (requires encoding)                         | Limited (one-hot for few categories)        | Good (optimal split finding)               | Best (ordered target statistics, no encoding)   |
| **Handling Missing Values**    | Yes (native)                                   | Yes                                         | Yes                                        | Yes                                             |
| **Regularization**             | Basic (shrinkage, subsampling)                 | Strong (L1/L2, max_depth, min_child_weight) | Basic + exclusive feature bundling         | Built-in (ordered boosting reduces overfitting) |
| **Sampling Techniques**        | Subsample, colsample_bytree (manual)           | Subsample, colsample                        | GOSS (Gradient-based One-Side Sampling)    | Ordered boosting + random permutations          |
| **GPU/Accelerated Support**    | No                                             | Yes (strong)                                | Yes (very strong)                          | Yes (excellent)                                 |
| **Feature Importance**         | Yes (gain, split count)                        | Yes (multiple types)                        | Yes                                        | Yes (highly accurate)                           |
| **Accuracy**                   | Good baseline                                  | High                                        | High (often slightly better on large data) | Very High (esp. with categoricals/noisy data)   |
| **Overfitting Resistance**     | Moderate                                       | Good (with tuning)                          | Moderate (leaf-wise can overfit)           | Excellent (due to ordered boosting)             |
| **Ease of Use / Tuning**       | Simple API                                     | Many hyperparameters                        | Fast defaults, fewer needed                | Very user-friendly (good defaults)              |
| **Weaknesses**                 | Slow on large data, no GPU                     | Steeper learning curve                      | Can overfit if not limited depth           | Slightly higher memory on CPU                   |
| **Use Cases**                  | Small-medium data, prototyping                 | General purpose, Kaggle competitions        | Very large datasets, speed critical        | Datasets with many categoricals, minimal prep   |

### Boosting

```mermaid
flowchart LR
    A(Start with equal weights on data) e1@--> B(Train weak learner on weighted data)
    B e2@--> C(Compute weighted error of learner)
    C e3@--> D(Calculate learner weight based on error)
    D e4@--> E(Update data weights: increase misclassified, decrease correct)
    E e5@--> F{More iterations?}
    F e6@-->|Yes| B
    F e7@-->|No| G(Combine weak learners with weights)
    G e8@--> H(Final strong learner)

    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
```

### Stacking

```mermaid
flowchart LR
    A(Training Data) e1@--> B1(Base Model 1)
    A e2@--> B2(Base Model 2)
    A e3@--> B3(Base Model 3)
    B1 e4@--> C(Generate Predictions<br/>via Cross-Validation)
    B2 e5@--> C
    B3 e6@--> C
    C e7@--> D(Train Meta-Model<br/>on Predictions)
    D e8@--> E(Final Prediction<br/>on New Data)

    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }

```
